In [3]:

# CELL 1: GPU CHECK

import os
import random
import shutil
import xml.etree.ElementTree as ET
import cv2
import numpy as np
import torch
from ultralytics import YOLO


print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU. Training will be slow.")

print(f"\nCurrent working folder (everything will save here): {os.getcwd()}")

VALID_EXTS = (".jpg", ".jpeg", ".png")
RANDOM_SEED = 42
CRACK_CODES = {"D00", "D10", "D20"}  # only these three count as "Crack"

PyTorch Version: 2.11.0+cu128
CUDA Available: True
Active GPU: NVIDIA GeForce RTX 5080 Laptop GPU

Current working folder (everything will save here): c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3


In [4]:

# CELL 2: CONVERT JAPAN + UNITED STATES INTO CRACK-ONLY YOLO FORMAT


RAW_JAPAN_DIR = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\final5\final6\final7\crack\crack3\dataset\Japan\Japan"
RAW_US_DIR = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\final5\final6\final7\crack\crack3\dataset\United_States\United_States"

CLEAN_JAPAN_DIR = os.path.join(os.getcwd(), "datasets_processed", "japan_crack_only")
CLEAN_US_DIR = os.path.join(os.getcwd(), "datasets_processed", "us_crack_only")

SPLIT_RATIOS = {"train": 0.8, "valid": 0.1}  # test comes from the country's own test folder


def convert_one_xml(xml_path, img_width, img_height):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    yolo_lines = []
    for obj in root.findall("object"):
        name = obj.find("name").text.strip()
        if name not in CRACK_CODES:
            continue
        box = obj.find("bndbox")
        xmin = float(box.find("xmin").text)
        ymin = float(box.find("ymin").text)
        xmax = float(box.find("xmax").text)
        ymax = float(box.find("ymax").text)
        x_center = ((xmin + xmax) / 2) / img_width
        y_center = ((ymin + ymax) / 2) / img_height
        width = (xmax - xmin) / img_width
        height = (ymax - ymin) / img_height
        yolo_lines.append(f"0 {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    return yolo_lines


def find_annotations_dir(split_dir):
    """RDD2022 sometimes nests XML files under an extra 'xmls' folder -
    this checks both possible locations."""
    candidates = [os.path.join(split_dir, "annotations", "xmls"),
                  os.path.join(split_dir, "annotations")]
    for c in candidates:
        if os.path.isdir(c) and any(f.endswith(".xml") for f in os.listdir(c)):
            return c
    return candidates[0]  # fall back - will just mean no annotations found


def process_country(raw_dir, clean_dir, country_label):
    #  train folder becomes train + valid
    train_split_dir = os.path.join(raw_dir, "train")
    train_img_dir = os.path.join(train_split_dir, "images")
    train_ann_dir = find_annotations_dir(train_split_dir)

    all_train_images = [f for f in os.listdir(train_img_dir) if f.lower().endswith(VALID_EXTS)]
    random.Random(RANDOM_SEED).shuffle(all_train_images)

    n_total = len(all_train_images)
    n_valid = int(n_total * SPLIT_RATIOS["valid"])
    assignments = ([("valid", f) for f in all_train_images[:n_valid]]
                   + [("train", f) for f in all_train_images[n_valid:]])

    print(f"{country_label}: {n_total} train-folder images -> "
          f"{n_total - n_valid} train / {n_valid} valid")

    for output_split, fname in assignments:
        img_path = os.path.join(train_img_dir, fname)
        image = cv2.imread(img_path)
        if image is None:
            continue
        h, w = image.shape[:2]

        xml_path = os.path.join(train_ann_dir, os.path.splitext(fname)[0] + ".xml")
        lines = convert_one_xml(xml_path, w, h) if os.path.exists(xml_path) else []

        dst_img_dir = os.path.join(clean_dir, output_split, "images")
        dst_lbl_dir = os.path.join(clean_dir, output_split, "labels")
        os.makedirs(dst_img_dir, exist_ok=True)
        os.makedirs(dst_lbl_dir, exist_ok=True)
        shutil.copy2(img_path, os.path.join(dst_img_dir, fname))
        with open(os.path.join(dst_lbl_dir, os.path.splitext(fname)[0] + ".txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + ("\n" if lines else ""))

    # --- test folder becomes test (usually no annotations - that's fine, correctly negative) ---
    test_img_dir = os.path.join(raw_dir, "test", "images")
    if os.path.exists(test_img_dir):
        test_ann_dir = find_annotations_dir(os.path.join(raw_dir, "test"))
        dst_img_dir = os.path.join(clean_dir, "test", "images")
        dst_lbl_dir = os.path.join(clean_dir, "test", "labels")
        os.makedirs(dst_img_dir, exist_ok=True)
        os.makedirs(dst_lbl_dir, exist_ok=True)

        for fname in os.listdir(test_img_dir):
            if not fname.lower().endswith(VALID_EXTS):
                continue
            img_path = os.path.join(test_img_dir, fname)
            image = cv2.imread(img_path)
            if image is None:
                continue
            h, w = image.shape[:2]
            xml_path = os.path.join(test_ann_dir, os.path.splitext(fname)[0] + ".xml")
            lines = convert_one_xml(xml_path, w, h) if os.path.exists(xml_path) else []
            shutil.copy2(img_path, os.path.join(dst_img_dir, fname))
            with open(os.path.join(dst_lbl_dir, os.path.splitext(fname)[0] + ".txt"), "w", encoding="utf-8") as f:
                f.write("\n".join(lines) + ("\n" if lines else ""))

    print(f"{country_label}: written to {clean_dir}")


process_country(RAW_JAPAN_DIR, CLEAN_JAPAN_DIR, "Japan")
process_country(RAW_US_DIR, CLEAN_US_DIR, "United States")


Japan: 10506 train-folder images -> 9456 train / 1050 valid
Japan: written to c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\datasets_processed\japan_crack_only
United States: 4805 train-folder images -> 4325 train / 480 valid
United States: written to c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\datasets_processed\us_crack_only


In [5]:

# CELL 3: MAKE SYNTHETIC "NIGHTTIME" VERSIONS OF THE TRAIN IMAGES


def make_it_look_like_night(image):
    darker = cv2.convertScaleAbs(image, alpha=0.35, beta=-30)
    gamma = 2.2
    lookup_table = np.array([((i / 255.0) ** gamma) * 255 for i in range(256)]).astype("uint8")
    darker = cv2.LUT(darker, lookup_table)
    b, g, r = cv2.split(darker)
    b = cv2.add(b, 15)
    darker = cv2.merge([b, g, r])
    return darker


def create_synthetic_night_copy(source_dataset_dir, output_dir, country_label):
    src_img_dir = os.path.join(source_dataset_dir, "train", "images")
    src_lbl_dir = os.path.join(source_dataset_dir, "train", "labels")
    if not os.path.exists(src_img_dir):
        return 0

    dst_img_dir = os.path.join(output_dir, "train", "images")
    dst_lbl_dir = os.path.join(output_dir, "train", "labels")
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)

    count = 0
    for fname in os.listdir(src_img_dir):
        if not fname.lower().endswith(VALID_EXTS):
            continue
        image = cv2.imread(os.path.join(src_img_dir, fname))
        if image is None:
            continue
        night_version = make_it_look_like_night(image)

        new_fname = f"synthnight_{country_label}_{fname}"
        cv2.imwrite(os.path.join(dst_img_dir, new_fname), night_version)

        src_lbl = os.path.join(src_lbl_dir, os.path.splitext(fname)[0] + ".txt")
        dst_lbl = os.path.join(dst_lbl_dir, os.path.splitext(new_fname)[0] + ".txt")
        if os.path.exists(src_lbl):
            shutil.copy2(src_lbl, dst_lbl)
        else:
            open(dst_lbl, "w").close()
        count += 1

    return count


SYNTH_NIGHT_DIR = os.path.join(os.getcwd(), "datasets_processed", "synthetic_night_crack")

count_japan = create_synthetic_night_copy(CLEAN_JAPAN_DIR, SYNTH_NIGHT_DIR, "japan")
print(f"Created {count_japan} synthetic-night images from Japan")

count_us = create_synthetic_night_copy(CLEAN_US_DIR, SYNTH_NIGHT_DIR, "us")
print(f"Created {count_us} synthetic-night images from United States")

print(f"\nTotal synthetic-night train images: {count_japan + count_us}")
print(f"Written to: {SYNTH_NIGHT_DIR}")

Created 9456 synthetic-night images from Japan
Created 4325 synthetic-night images from United States

Total synthetic-night train images: 13781
Written to: c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\datasets_processed\synthetic_night_crack


In [6]:

# CELL 4: FIND ALL DATASETS AUTOMATICALLY 

BASE_DIR = os.path.join(os.getcwd(), "datasets_processed")
SPLIT_NAMES = {"train", "valid", "test"}

DATASETS = {}

for root, dirs, files in os.walk(BASE_DIR):
    if os.path.basename(root) != "images":
        continue
    split_dir = os.path.dirname(root)
    split_name = os.path.basename(split_dir)
    dataset_dir = os.path.dirname(split_dir)
    if split_name not in SPLIT_NAMES:
        continue
    labels_dir = os.path.join(split_dir, "labels")
    dataset_name = os.path.relpath(dataset_dir, BASE_DIR)
    DATASETS.setdefault(dataset_name, {})[split_name] = (root, labels_dir)

print(f"Auto-discovered {len(DATASETS)} datasets:")
for name, splits in DATASETS.items():
    print(f"  {name}: {sorted(splits.keys())}")

Auto-discovered 3 datasets:
  japan_crack_only: ['test', 'train', 'valid']
  synthetic_night_crack: ['train']
  us_crack_only: ['test', 'train', 'valid']


In [7]:

# CELL 5: FIND WHICH IMAGES ARE ANNOTATED (POSITIVE VS NEGATIVE)

def scan_split(images_dir, labels_dir):
    positive, negative = [], []
    if not os.path.exists(images_dir):
        return positive, negative
    for fname in os.listdir(images_dir):
        if not fname.lower().endswith(VALID_EXTS):
            continue
        img_path = os.path.join(images_dir, fname)
        label_path = os.path.join(labels_dir, os.path.splitext(fname)[0] + ".txt")
        has_content = False
        if os.path.exists(label_path):
            with open(label_path, "r", encoding="utf-8") as f:
                has_content = any(line.strip() for line in f)
        if has_content:
            positive.append(img_path)
        else:
            negative.append(img_path)
    return positive, negative


scan_results = {}
print(f"{'Dataset':<25}{'Split':<8}{'Positive':<10}{'Negative':<10}{'Total':<8}")

for ds_name, splits in DATASETS.items():
    scan_results[ds_name] = {}
    for split_name, (img_dir, lbl_dir) in splits.items():
        pos, neg = scan_split(img_dir, lbl_dir)
        scan_results[ds_name][split_name] = (pos, neg)
        print(f"{ds_name:<25}{split_name:<8}{len(pos):<10}{len(neg):<10}{len(pos)+len(neg):<8}")

Dataset                  Split   Positive  Negative  Total   
japan_crack_only         test    0         2627      2627    
japan_crack_only         train   6679      2777      9456    
japan_crack_only         valid   749       301       1050    
synthetic_night_crack    train   10981     2800      13781   
us_crack_only            test    0         1200      1200    
us_crack_only            train   4302      23        4325    
us_crack_only            valid   479       1         480     


In [8]:

# CELL 6: BUILD COMBINED TRAIN/VAL/TEST LISTS

OUTPUT_DIR = os.path.join(os.getcwd(), "combined_lists")
os.makedirs(OUTPUT_DIR, exist_ok=True)

def write_list(split_name, out_filename):
    all_paths = []
    for ds_name in DATASETS:
        pos, neg = scan_results[ds_name].get(split_name, ([], []))
        all_paths.extend(pos)
        all_paths.extend(neg)
    out_path = os.path.join(OUTPUT_DIR, out_filename)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(all_paths))
    print(f"{split_name}: wrote {len(all_paths)} image paths -> {out_path}")
    return out_path

train_txt = write_list("train", "train.txt")
valid_txt = write_list("valid", "valid.txt")
test_txt  = write_list("test", "test.txt")

train: wrote 27562 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\train.txt
valid: wrote 1530 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\valid.txt
test: wrote 3827 image paths -> c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\test.txt


In [9]:

# CELL 7: WRITE THE FINAL data.yaml

import yaml

data_yaml_path = os.path.join(OUTPUT_DIR, "crack3_data.yaml")

data_yaml = {
    "train": train_txt,
    "val": valid_txt,
    "test": test_txt,
    "nc": 1,
    "names": ["Crack"],
}

with open(data_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print(f"data.yaml written to: {data_yaml_path}")

data.yaml written to: c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\crack3_data.yaml


In [10]:

# CELL 8: CLASS VERIFICATION

def verify_classes():
    invalid = 0
    checked = 0
    for ds_name in DATASETS:
        for split_name in ["train", "valid", "test"]:
            if split_name not in DATASETS[ds_name]:
                continue
            pos, _ = scan_results[ds_name].get(split_name, ([], []))
            lbl_dir = DATASETS[ds_name][split_name][1]
            for img_path in pos:
                lbl_path = os.path.join(lbl_dir, os.path.splitext(os.path.basename(img_path))[0] + ".txt")
                with open(lbl_path, "r", encoding="utf-8") as f:
                    for line in f:
                        if not line.strip():
                            continue
                        checked += 1
                        if int(line.split()[0]) != 0:
                            invalid += 1

    print(f"Checked {checked} annotations across all datasets.")
    if invalid == 0:
        print("Verification passed: every annotation is class 0 (Crack).")
    else:
        print(f"WARNING: found {invalid} annotations with an unexpected class id!")

verify_classes()

Checked 47655 annotations across all datasets.
Verification passed: every annotation is class 0 (Crack).


In [ ]:

# CELL 9: TRAINING (JAPAN + US + SYNTHETIC NIGHT, 800px)


PROJECT_NAME = "Crack3_Runs"

model = YOLO("yolo26m.pt")

print("Starting training - Crack 3 (Japan + US + synthetic night, 800px)...")

results = model.train(
    data=data_yaml_path,
    epochs=100,
    patience=15,
    imgsz=800,

    device=0,
    cache="disk",
    workers=6,
    batch=16,

    optimizer="AdamW",
    lr0=0.0001,
    cos_lr=False,
    warmup_epochs=3,

    hsv_v=0.4,
    hsv_s=0.3,
    mosaic=1.0,
    fliplr=0.5,

    project=PROJECT_NAME,
    name="crack3",
)

best_model_path = os.path.join("runs", "detect", PROJECT_NAME, "crack3", "weights", "best.pt")
print(f"\nTraining finished. Best weights saved at: {best_model_path}")

Starting training - Crack 3 (Japan + US + synthetic night, 800px)...
New https://pypi.org/project/ultralytics/8.4.139 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\crack3_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, 

In [11]:
# CELL 10: CRACK 3 VALIDATION RESULTS

from ultralytics import YOLO

best_model_path = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.pt"

data_yaml_path = r"C:\Users\akaka\Desktop\msc thesis\final 2\final 3\final 4\final5\final6\final7\crack\crack3\combined_lists\crack3_data.yaml"

final_model = YOLO(best_model_path)

metrics = final_model.val(
    data=data_yaml_path,
    split="val",
    imgsz=800,
    device=0
)

print("\nCRACK 3 VALIDATION RESULTS")
print(f"Precision  : {metrics.box.mp:.3f}")
print(f"Recall     : {metrics.box.mr:.3f}")
print(f"mAP50      : {metrics.box.map50:.3f}")
print(f"mAP50-95   : {metrics.box.map:.3f}")

Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080 Laptop GPU, 16275MiB)
YOLO26m summary (fused): 132 layers, 20,350,223 parameters, 0 gradients, 68.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 19.94.2 MB/s, size: 75.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\datasets_processed\japan_crack_only\valid\labels.cache... 1530 images, 302 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1530/1530  0.0s
val: c:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\datasets_processed\japan_crack_only\valid\images\Japan_007271.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 96/96 3.9it/s 24.4s0.3ss
                   all       1530  

In [13]:

# CELL 11: SPEED BENCHMARK (FPS)

import time

with open(test_txt, "r", encoding="utf-8") as f:
    all_test_images = [line.strip() for line in f if line.strip()]

benchmark_images = all_test_images[:100] if len(all_test_images) >= 100 else all_test_images
print(f"Benchmarking speed on {len(benchmark_images)} images...")

_ = final_model.predict(source=benchmark_images[0], imgsz=800, verbose=False)

start_time = time.time()
for img_path in benchmark_images:
    _ = final_model.predict(source=img_path, imgsz=800, verbose=False)
elapsed = time.time() - start_time

fps = len(benchmark_images) / elapsed
print(f"\nProcessed {len(benchmark_images)} images in {elapsed:.2f} seconds")
print(f"Speed: {fps:.1f} FPS")

Benchmarking speed on 100 images...

Processed 100 images in 1.32 seconds
Speed: 75.9 FPS


In [15]:

# CELL 12: LIVE WEBCAM DETECTION (CRACK ONLY)


CAMERA_INDEX = 1  # 0 = laptop webcam, 1 = phone via Phone Link

cap = cv2.VideoCapture(CAMERA_INDEX, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Could not open camera. Check CAMERA_INDEX, or that Phone Link is connected.")
else:
    print("Camera started. Click the video window and press 'q' to stop.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = final_model.predict(frame, imgsz=800, conf=0.35, verbose=False)

        for result in results:
            if result.boxes is None:
                continue
            for box in result.boxes:
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                conf = float(box.conf[0])
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
                cv2.putText(frame, f"Crack {conf:.2f}", (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

        cv2.imshow("Crack Detection - press q to quit", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()
    print("Camera stopped.")

Camera started. Click the video window and press 'q' to stop.
Camera stopped.


In [16]:
# CELL 13: COMPRESS THE MODEL (FP16 ONNX)

import os

print("Exporting Crack 3 to FP16 ONNX...")

compressed_model_path = final_model.export(
    format="onnx",
    half=True,
    imgsz=800,
    simplify=True
)

original_size = os.path.getsize(best_model_path) / (1024 * 1024)
compressed_size = os.path.getsize(compressed_model_path) / (1024 * 1024)

print(f"Original size:   {original_size:.1f} MB")
print(f"Compressed size: {compressed_size:.1f} MB")
print(f"Total Reduction: {(1 - compressed_size/original_size) * 100:.1f}%")
print(f"Saved to: {compressed_model_path}")

Exporting Crack 3 to FP16 ONNX...
WARNING 'half' is deprecated and will be removed in the future. Use 'quantize' instead.
Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CPU (Intel Core Ultra 9 275HX)

PyTorch: starting from 'C:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.pt' with input shape (1, 3, 800, 800) BCHW and output shape(s) (1, 300, 6) (42.0 MB)

ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
ONNX: converting to FP16...
ONNX: export success  3.6s, saved as 'C:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.onnx' (39.1 MB)

Export complete (4.1s)
Results saved to C:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.onnx
Predict:         yolo predict task=detect model=C:\Users\akaka\Desktop\final 2\final 3\f

In [17]:
# CELL 14: VALIDATE THE COMPRESSED MODEL

compressed_model = YOLO(compressed_model_path)

comp_metrics = compressed_model.val(
    data=data_yaml_path,
    split="val",
    imgsz=800,
    device="cpu"
)

print("\n--- ORIGINAL vs COMPRESSED ---")
print(f"{'Metric':<12}{'Original':<12}{'Compressed':<12}{'Difference':<12}")

print(f"{'Precision':<12}{metrics.box.mp:<12.3f}{comp_metrics.box.mp:<12.3f}{comp_metrics.box.mp - metrics.box.mp:+.3f}")
print(f"{'Recall':<12}{metrics.box.mr:<12.3f}{comp_metrics.box.mr:<12.3f}{comp_metrics.box.mr - metrics.box.mr:+.3f}")
print(f"{'mAP50':<12}{metrics.box.map50:<12.3f}{comp_metrics.box.map50:<12.3f}{comp_metrics.box.map50 - metrics.box.map50:+.3f}")
print(f"{'mAP50-95':<12}{metrics.box.map:<12.3f}{comp_metrics.box.map:<12.3f}{comp_metrics.box.map - metrics.box.map:+.3f}")

Ultralytics 8.4.135  Python-3.12.10 torch-2.11.0+cu128 CPU (Intel Core Ultra 9 275HX)
Loading C:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING requirements:  Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\

In [18]:
# CELL 15: SPEED BENCHMARK - COMPRESSED MODEL

import time

with open(test_txt, "r", encoding="utf-8") as f:
    all_test_images = [line.strip() for line in f if line.strip()]

benchmark_images = all_test_images[:100]

print(f"Benchmarking COMPRESSED model speed on {len(benchmark_images)} images...")

# Warm-up
_ = compressed_model.predict(
    source=benchmark_images[0],
    imgsz=800,
    verbose=False,
    device="cpu"
)

start_time = time.time()

for img_path in benchmark_images:
    _ = compressed_model.predict(
        source=img_path,
        imgsz=800,
        verbose=False,
        device="cpu"
    )

elapsed = time.time() - start_time

compressed_fps = len(benchmark_images) / elapsed

print(f"\nProcessed {len(benchmark_images)} images in {elapsed:.2f} seconds")
print(f"Compressed model speed: {compressed_fps:.1f} FPS")

Benchmarking COMPRESSED model speed on 100 images...
Loading C:\Users\akaka\Desktop\final 2\final 3\final 4\final5\final6\final7\crack\crack3\runs\detect\Crack3_Runs\crack3\weights\best.onnx for ONNX Runtime inference...
requirements: Ultralytics requirement ['onnxruntime'] not found, attempting AutoUpdate...
WARNING Retry 1/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING Retry 2/2 failed: Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3\\.venv\\Scripts\\python.exe', 'onnxruntime', '--index-strategy=unsafe-best-match', '--break-system-packages']' returned non-zero exit status 1.
WARNING requirements:  Command '['uv', 'pip', 'install', '--no-cache-dir', '--python', 'c:\\Users\\akaka\\Desktop\\final 2\\final 3